[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/04_Feature_Types_and_Target_Influence.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/04_Feature_Types_and_Target_Influence.ipynb)

# How Each Feature Type Influences the Target — One Observation at a Time

A real dataset never has just one type of feature. You find:

- **Continuous** numbers: age (38), income (75k), credit score (680)
- **Discrete** counts: number of children (0, 1, 2, 3 …)
- **Ordinal** steps: education level (1=HS → 4=PhD)
- **Binary** flags: owns a home? (yes=1 / no=0)
- **Categorical** labels: city (urban / suburban / rural)

Each type carries information differently, is correlated with the target differently,
and contributes to the prediction in its own way.

---

**Four questions this notebook answers**

1. How do you measure correlation with the target for each feature type?
2. For one observation — which features are loud, and which are completely silent?
3. Why does a binary feature with value 0 contribute absolutely nothing?
4. For a categorical feature — which encoded column actually speaks?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)
print('Ready.')

In [ ]:
n = 300

# ── Continuous ──────────────────────────────────────────────
age          = np.random.randint(25, 60, n).astype(float)
income       = np.random.uniform(30, 150, n)      # thousands (30k–150k)
credit_score = np.random.uniform(550, 820, n)

# ── Discrete (whole-number count) ───────────────────────────
num_children = np.random.choice([0,1,2,3,4], n, p=[0.30,0.30,0.22,0.13,0.05])

# ── Ordinal (ordered levels, gap may not be equal) ──────────
education = np.random.choice([1,2,3,4], n, p=[0.20,0.40,0.30,0.10])
#   1=High School  2=Bachelor  3=Master  4=PhD

# ── Binary (two states only) ────────────────────────────────
has_prior_loan = np.random.choice([0,1], n, p=[0.45,0.55])
is_homeowner   = np.random.choice([0,1], n, p=[0.50,0.50])

# ── Categorical → one-hot (urban is the baseline: both dummies=0) ──
city          = np.random.choice(['urban','suburban','rural'], n, p=[0.50,0.30,0.20])
city_suburban = (city == 'suburban').astype(float)
city_rural    = (city == 'rural').astype(float)

# ── Known true weights ───────────────────────────────────────
TRUE_INTERCEPT = 10.0
TRUE_W = {
    'age':            0.30,   # each extra year    → +0.30k credit limit
    'income':         0.15,   # each extra $1k     → +0.15k
    'credit_score':   0.05,   # each extra point   → +0.05k
    'num_children':  -2.00,   # each extra child   → -2.00k
    'education':      4.00,   # each level up      → +4.00k
    'has_prior_loan': 8.00,   # repaid before      → +8.00k
    'is_homeowner':  12.00,   # owns home          → +12.00k
    'city_suburban':  3.00,   # suburban vs urban  → +3.00k
    'city_rural':    -5.00,   # rural vs urban     → -5.00k
}

credit_limit = (
    TRUE_INTERCEPT
    + TRUE_W['age']            * age
    + TRUE_W['income']         * income
    + TRUE_W['credit_score']   * credit_score
    + TRUE_W['num_children']   * num_children
    + TRUE_W['education']      * education
    + TRUE_W['has_prior_loan'] * has_prior_loan
    + TRUE_W['is_homeowner']   * is_homeowner
    + TRUE_W['city_suburban']  * city_suburban
    + TRUE_W['city_rural']     * city_rural
    + np.random.normal(0, 5, n)
)

FEATURES = ['age','income','credit_score','num_children','education',
            'has_prior_loan','is_homeowner','city_suburban','city_rural']

df = pd.DataFrame({
    'age': age, 'income': income, 'credit_score': credit_score,
    'num_children': num_children, 'education': education,
    'has_prior_loan': has_prior_loan, 'is_homeowner': is_homeowner,
    'city': city, 'city_suburban': city_suburban, 'city_rural': city_rural,
    'credit_limit': credit_limit
})

print(f"Dataset: {n} loan applicants, 9 features spanning all four types")
print(f"Target (credit limit): {credit_limit.min():.0f}k to {credit_limit.max():.0f}k")
df[['age','income','credit_score','num_children','education',
    'has_prior_loan','is_homeowner','city','credit_limit']].head(5)

## Part 1 — Feature Types: Four Ways Data Encodes Information

The type of a feature controls:
- which **correlation measure** is appropriate
- how the **feature value enters the prediction** for one observation
- when the feature is **completely silent** for one row

| Feature | Type | Possible values | Key property |
|---|---|---|---|
| `age` | Continuous | 25, 38.5, 59 … | Any value in a range; distance between values is meaningful |
| `income` | Continuous | 30k – 150k | Scale matters — larger values carry more weight in raw gradients |
| `credit_score` | Continuous | 550 – 820 | Any value in range |
| `num_children` | Discrete | 0, 1, 2, 3, 4 | Whole numbers only; count of something |
| `education` | Ordinal | 1, 2, 3, 4 | Ordered steps; gap between levels may not be equal |
| `has_prior_loan` | Binary | 0 or 1 | Only two states — absent or present |
| `is_homeowner` | Binary | 0 or 1 | Only two states |
| `city_suburban` | Categorical (dummy) | 0 or 1 | Encodes one level of a 3-level category |
| `city_rural` | Categorical (dummy) | 0 or 1 | Encodes one level; urban = baseline (both=0) |

In [ ]:
# We will use observation 42 as our example throughout the notebook
OBS = 42
obs = df.iloc[OBS]

edu_label  = {1:'HS', 2:'Bachelor', 3:'Master', 4:'PhD'}
type_label = {
    'age':            'Continuous',
    'income':         'Continuous',
    'credit_score':   'Continuous',
    'num_children':   'Discrete',
    'education':      'Ordinal',
    'has_prior_loan': 'Binary',
    'is_homeowner':   'Binary',
    'city':           'Categorical',
}

print(f"=== Observation {OBS} ===")
print()
for feat, ftype in type_label.items():
    val = obs[feat]
    if feat == 'education':
        note = f"{val:.0f}  ({edu_label[int(val)]})"
    elif feat == 'income':
        note = f"{val:.1f}k  (${val*1000:,.0f})"
    elif feat in ('has_prior_loan', 'is_homeowner'):
        note = f"{val:.0f}  ({'YES — active' if val==1 else 'NO  — silent'})"
    elif feat == 'city':
        note = f"{val}  (suburban=1, rural=0)"
    else:
        note = f"{val:.1f}"
    print(f"  {feat:<18}  [{ftype:<20}]   {note}")

print()
print(f"  {'credit_limit':<18}  [TARGET               ]   {obs['credit_limit']:.1f}k")

## Part 2 — Population Correlation: Different Measure for Each Feature Type

We compute correlation over all 300 rows to learn which features matter.

**Why different measures?**

- **Pearson r** assumes a *linear* relationship between two continuous numbers
- **Spearman ρ** uses *ranks* only — works for ordinal and discrete features where the gap between levels may not be uniform
- **Point-biserial r** is Pearson applied when one variable is binary (0/1) — it measures how different the target mean is between the two groups
- **One-hot Pearson** for categorical dummies — compute Pearson separately for each dummy column

All four measures return a value from −1 to +1.
Near 0 = weak relationship. Near ±1 = strong relationship.

In [ ]:
results = {}

# Continuous: Pearson (linear relationship)
for feat in ['age', 'income', 'credit_score']:
    r, _ = stats.pearsonr(df[feat], df['credit_limit'])
    results[feat] = ('Pearson r', round(r, 3))

# Discrete + ordinal: Spearman (rank-based, handles unequal gaps)
for feat in ['num_children', 'education']:
    r, _ = stats.spearmanr(df[feat], df['credit_limit'])
    results[feat] = ('Spearman rho', round(r, 3))

# Binary: point-biserial (same math as Pearson, named differently)
for feat in ['has_prior_loan', 'is_homeowner']:
    r, _ = stats.pointbiserialr(df[feat], df['credit_limit'])
    results[feat] = ('Point-biserial', round(r, 3))

# Categorical dummies: Pearson per dummy column
for feat in ['city_suburban', 'city_rural']:
    r, _ = stats.pearsonr(df[feat], df['credit_limit'])
    results[feat] = ('Pearson r (dummy)', round(r, 3))

corr_df = pd.DataFrame(
    [{'feature': f, 'measure': m, 'r': v} for f, (m, v) in results.items()]
)

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#4CAF50' if v > 0 else '#e57373' for v in corr_df['r']]
bars = ax.barh(corr_df['feature'], corr_df['r'], color=bar_colors)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Correlation with credit_limit')
ax.set_title('Population-level correlation — appropriate measure for each feature type\n'
             'Green = positive relationship   Red = negative relationship', fontsize=11)

for bar, (_, row) in zip(bars, corr_df.iterrows()):
    x  = row['r']
    ha = 'left' if x >= 0 else 'right'
    ax.text(x + 0.01 if x >= 0 else x - 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{row['measure']}: {x:.3f}",
            va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.show()

print()
print("education:      Spearman = high positive → higher education level → higher credit limit")
print("is_homeowner:   Point-biserial = high positive → homeowners get much larger limits")
print("num_children:   Spearman = negative → more children → lower limit (less disposable income)")
print("city_rural:     negative → rural applicants get lower limits than urban")

## Part 3 — Train a Model and Learn the Weights

We train a Linear Regression on **scaled features**.

Scaling is essential before training because features are on completely different scales:
- `credit_score` ranges 550–820
- `is_homeowner` ranges 0–1

Without scaling, credit_score would dominate the gradient purely because of its larger numbers,
not because it is more important. StandardScaler brings all features to mean=0, std=1.

After training, we can recover the **raw-space weights** by un-scaling:
`raw_weight_j = scaled_weight_j / std_j`

These raw weights are what we multiply by raw feature values to get contributions.

In [ ]:
X = df[FEATURES].values
y = df['credit_limit'].values

scaler   = StandardScaler()
X_sc     = scaler.fit_transform(X)

model = LinearRegression().fit(X_sc, y)

# Convert scaled weights back to raw feature space for interpretable contributions
raw_weights = model.coef_ / scaler.scale_

print("True weights (used to generate data) vs. Learned weights")
print()
print(f"{'Feature':<18}  {'True':>8}  {'Learned':>8}  {'Match?'}")
print("-" * 50)
for feat, tw, lw in zip(FEATURES, [TRUE_W[f] for f in FEATURES], raw_weights):
    match = 'OK' if abs(tw - lw) < abs(tw) * 0.2 else 'close'
    print(f"{feat:<18}  {tw:>8.2f}  {lw:>8.2f}  {match}")

print()
print("The model recovered weights close to the true values.")
print("Small differences come from noise in the data (sd=5k added during generation).")

## Part 4 — One Observation: Contribution = Weight × Feature Value

For observation 42, the model's prediction is:

```
prediction = intercept
           + weight_age           × value_age
           + weight_income        × value_income
           + weight_credit_score  × value_credit_score
           + weight_num_children  × value_num_children
           + weight_education     × value_education
           + weight_has_prior_loan × value_has_prior_loan
           + weight_is_homeowner  × value_is_homeowner
           + weight_city_suburban × value_city_suburban
           + weight_city_rural    × value_city_rural
```

Each term `weight_j × value_j` is the **contribution** of feature j for this specific person.

The key rule: **if the feature value is zero, the contribution is exactly zero — the feature is silent.**

For continuous features, the value is almost never exactly zero, so they almost always contribute.
For binary and categorical (one-hot) features, value=0 is common and means silence.

In [ ]:
obs_vals = df[FEATURES].iloc[OBS].values
contrib  = raw_weights * obs_vals

contrib_df = pd.DataFrame({
    'feature':      FEATURES,
    'ftype':        ['Continuous','Continuous','Continuous','Discrete','Ordinal',
                     'Binary','Binary','Categorical','Categorical'],
    'value':        obs_vals,
    'weight':       raw_weights,
    'contribution': contrib,
})

print(f"Observation {OBS}  |  city = {df['city'].iloc[OBS]}  |  Actual limit = {y[OBS]:.1f}k")
print()
print(f"  {'Feature':<18} {'Type':<12} {'Value':>8} {'Weight':>8} {'Contribution':>14}")
print("  " + "-" * 64)
for _, row in contrib_df.iterrows():
    silent = '  ← SILENT' if row['value'] == 0 else ''
    print(f"  {row['feature']:<18} {row['ftype']:<12} {row['value']:>8.2f}"
          f" {row['weight']:>8.2f} {row['contribution']:>12.2f}k{silent}")

print("  " + "-" * 64)
print(f"  {'Intercept':<18} {'':>12} {'':>8} {'':>8} {TRUE_INTERCEPT:>12.2f}k")
total = TRUE_INTERCEPT + contrib.sum()
print(f"  {'TOTAL':<18} {'':>12} {'':>8} {'':>8} {total:>12.2f}k")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

bar_colors = []
for v, c in zip(obs_vals, contrib):
    if v == 0:
        bar_colors.append('#cccccc')    # gray  = silent (value is zero)
    elif c > 0:
        bar_colors.append('#4CAF50')    # green = pushes prediction up
    else:
        bar_colors.append('#e57373')    # red   = pushes prediction down

bars = ax.barh(FEATURES, contrib, color=bar_colors, edgecolor='white', height=0.6)

for bar, (_, row) in zip(bars, contrib_df.iterrows()):
    v = row['value']
    w = row['weight']
    c = row['contribution']
    y_mid = bar.get_y() + bar.get_height() / 2
    if v == 0:
        ax.text(0.5, y_mid,
                f"value = 0  →  {w:.2f} x 0 = 0   SILENT",
                va='center', ha='left', fontsize=8, color='#888888', style='italic')
    else:
        xpos = c + 0.5 if c >= 0 else c - 0.5
        ha   = 'left'  if c >= 0 else 'right'
        ax.text(xpos, y_mid,
                f"{w:.2f} x {v:.1f} = {c:.1f}k",
                va='center', ha=ha, fontsize=8)

ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Contribution to predicted credit limit (thousands $)')
ax.set_title(f'Feature contributions for observation {OBS}\n'
             'Gray = SILENT (feature value is 0)   '
             'Green = adds to prediction   Red = subtracts', fontsize=11)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Positive contribution'),
    Patch(facecolor='#e57373', label='Negative contribution'),
    Patch(facecolor='#cccccc', label='Silent — feature value is 0'),
], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

## Part 5 — Binary Features: Present = Speaks, Absent = Completely Silent

For a binary feature (0 or 1):

```
contribution = weight × 0 = 0      (feature is off — completely silent)
contribution = weight × 1 = weight  (feature is on  — full weight passes through)
```

The **weight** is a property of the model — it is the same for all applicants.
The **value** is a property of the specific person — it decides whether the weight matters at all.

This means for observation 42, whether `is_homeowner = 0` receives +12k or nothing
is entirely determined by that one bit (0 or 1).

In [ ]:
print("How is_homeowner switches a contribution on or off:")
print()

for scenario, val in [("Applicant A  (is_homeowner = 1)", 1),
                      ("Applicant B  (is_homeowner = 0)", 0)]:
    weight    = TRUE_W['is_homeowner']
    contrib_v = weight * val
    status    = "ACTIVE — weight passes through" if val == 1 else "SILENT — weight multiplied by 0"
    print(f"  {scenario}")
    print(f"    contribution = {weight:.1f} (weight)  x  {val}  (value)  =  {contrib_v:.1f}k")
    print(f"    Status: {status}")
    print()

print(f"  Difference from this one feature alone: {TRUE_W['is_homeowner']:.1f}k")
print()
print("  The weight is learned once from all the data.")
print("  The value decides, for each individual person, whether that learned knowledge")
print("  enters their prediction at all.")
print()

# Visualise with a simple diagram
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, (val, label) in zip(axes, [(1, 'is_homeowner = 1\n(owns a home)'),
                                    (0, 'is_homeowner = 0\n(does not own a home)')]):
    w = TRUE_W['is_homeowner']
    c = w * val
    color = '#4CAF50' if c > 0 else '#cccccc'
    ax.barh(['is_homeowner'], [c], color=color, height=0.4)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlim(-2, 15)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Contribution (k$)')
    note = f"{w:.0f} x {val} = {c:.0f}k" if c > 0 else f"{w:.0f} x {val} = 0k  (SILENT)"
    ax.text(0.5, 0, note, va='center', ha='left', fontsize=10,
            color='gray' if c == 0 else 'black', style='italic' if c == 0 else 'normal')

plt.suptitle('Same weight, different value — one applicant gets +12k, the other gets nothing', fontsize=11)
plt.tight_layout()
plt.show()

## Part 6 — Categorical Features: Only the Active Label Has a Voice

A categorical feature like `city` (urban / suburban / rural) cannot enter the model as a raw string.
It is converted to **one-hot encoding** — one binary column per category:

| city value | city_suburban | city_rural |
|---|---|---|
| urban | 0 | 0 ← **baseline** (both dummies off) |
| suburban | **1** | 0 |
| rural | 0 | **1** |

Urban is the **baseline** — when both dummies are 0, no city adjustment is applied.
The model learns one weight for `city_suburban` and one for `city_rural`.

For any single observation:
- exactly one category is active (value = 1) — or none if it is the baseline
- all other category columns are silent (value = 0)
- only the active column's weight enters the prediction

In [ ]:
city_weights = {
    'urban':    {'city_suburban': 0, 'city_rural': 0},   # baseline — both dummies off
    'suburban': {'city_suburban': 1, 'city_rural': 0},
    'rural':    {'city_suburban': 0, 'city_rural': 1},
}

print("City encoding and contribution for each possible city value:")
print()
print(f"  {'City':<12} {'city_suburban':>14} {'city_rural':>12} {'suburb contrib':>16} {'rural contrib':>14} {'total city':>12}")
print("  " + "-" * 74)

for city_val, dummies in city_weights.items():
    sub_val   = dummies['city_suburban']
    rur_val   = dummies['city_rural']
    sub_c     = TRUE_W['city_suburban'] * sub_val
    rur_c     = TRUE_W['city_rural']    * rur_val
    total_c   = sub_c + rur_c
    print(f"  {city_val:<12} {sub_val:>14}  {rur_val:>12} {sub_c:>14.1f}k {rur_c:>13.1f}k {total_c:>11.1f}k")

print()
print("  Urban applicants: both dummies = 0 → city contributes 0k (it IS the baseline)")
print("  Suburban:  city_suburban=1 → +3.0k    city_rural=0 → silent")
print("  Rural:     city_suburban=0 → silent    city_rural=1 → -5.0k")
print()

# Show as a bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, city_val in zip(axes, ['urban', 'suburban', 'rural']):
    dummies   = city_weights[city_val]
    sub_c     = TRUE_W['city_suburban'] * dummies['city_suburban']
    rur_c     = TRUE_W['city_rural']    * dummies['city_rural']
    vals      = [sub_c, rur_c]
    cols      = ['#cccccc' if v == 0 else ('#4CAF50' if v > 0 else '#e57373') for v in vals]
    bars      = ax.bar(['city_suburban', 'city_rural'], vals, color=cols, edgecolor='white')
    ax.axhline(0, color='black', lw=1)
    ax.set_ylim(-7, 5)
    ax.set_title(f"city = {city_val}", fontsize=11)
    ax.set_ylabel('Contribution (k$)')
    for bar, v in zip(bars, vals):
        label = f"{v:.0f}k" if v != 0 else "0k (SILENT)"
        ax.text(bar.get_x() + bar.get_width()/2, v + (0.2 if v >= 0 else -0.4),
                label, ha='center', fontsize=9,
                color='gray' if v == 0 else 'black', style='italic' if v == 0 else 'normal')

plt.suptitle('One-hot columns: only the active category speaks', fontsize=11, y=1.05)
plt.tight_layout()
plt.show()

## Part 7 — The Gradient: How One Observation Votes on Each Weight

During training, every observation pushes each weight in a direction:

```
gradient vote for weight j  =  error × feature_value_j
```

- `error` = how wrong the current prediction is for this observation
- `feature_value_j` = the lever — a larger value means a louder vote

**What this means for each feature type:**

| Feature type | When its gradient vote is zero | When it is loud |
|---|---|---|
| Continuous | Almost never (value rarely = 0) | Large values (credit_score=680, income=75) are loud |
| Binary | When value = 0 (feature is off) | When value = 1 — it votes exactly `error` |
| Categorical (dummy) | When category is inactive (value=0) | When category is active (value=1) |
| Discrete | When count = 0 | Proportional to the count |

**The scale problem**: without StandardScaler, income (75) and credit_score (680) shout over
is_homeowner (0 or 1) simply because their numbers are bigger — not because they are more important.

In [ ]:
# Train model already done above. Compute prediction and error for obs OBS.
pred_obs = model.predict(X_sc[OBS].reshape(1, -1))[0]
error    = y[OBS] - pred_obs

print(f"Observation {OBS}:")
print(f"  Actual credit limit : {y[OBS]:.2f}k")
print(f"  Model prediction    : {pred_obs:.2f}k")
print(f"  Error (actual-pred) : {error:.2f}k")
print()

# Raw gradient vote: error x raw feature value
raw_vals   = X[OBS]
grad_raw   = error * raw_vals

# Scaled gradient vote: error x z-score of feature value
sc_vals    = X_sc[OBS]
grad_sc    = error * sc_vals

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Raw ──────────────────────────────────────────────────────
colors_raw = ['#cccccc' if v == 0 else ('#4CAF50' if v > 0 else '#e57373') for v in grad_raw]
axes[0].barh(FEATURES, grad_raw, color=colors_raw, edgecolor='white')
axes[0].axvline(0, color='black', lw=1.2)
axes[0].set_title('Gradient vote — RAW feature values\n'
                  'credit_score and income dominate purely due to scale', fontsize=10)
axes[0].set_xlabel('error x raw value')
# annotate silent bars
for i, (feat, v) in enumerate(zip(FEATURES, grad_raw)):
    if v == 0:
        axes[0].text(1, i, 'SILENT', va='center', fontsize=8, color='gray', style='italic')

# ── Scaled ───────────────────────────────────────────────────
colors_sc = ['#cccccc' if abs(v) < 0.05 else ('#4CAF50' if v > 0 else '#e57373') for v in grad_sc]
axes[1].barh(FEATURES, grad_sc, color=colors_sc, edgecolor='white')
axes[1].axvline(0, color='black', lw=1.2)
axes[1].set_title('Gradient vote — SCALED feature values (z-scores)\n'
                  'all features compete on a fair scale', fontsize=10)
axes[1].set_xlabel('error x z-score')

plt.suptitle('How observation 42 pushes each weight during one training step\n'
             'Gray = silent (value=0 raw) or near-zero (scaled)',
             fontsize=11, y=1.03)
plt.tight_layout()
plt.show()

print("Raw:    credit_score gradient is ~680x larger than is_homeowner — just because of scale")
print("Scaled: each feature's vote is proportional to how far this person's value is from average")
print("        z-score = (value - mean) / std  — positive means above average, negative below")

## Part 8 — Discrete and Ordinal Features: Number or Category?

When you treat `num_children` as a number:
- the model assumes: 0 → 1 child adds the same effect as 1 → 2 children
- contribution = weight × count (e.g., weight × 2 for two children)

When you treat it as one-hot (like a categorical):
- the model learns a separate value for 0, 1, 2, 3, 4 children
- this is more flexible but needs more data to learn each level

**Same question for `education` (1, 2, 3, 4):**
- as a number: assumes the gap from HS→Bachelor is the same as Master→PhD
- as one-hot: no such assumption — but you lose the natural ordering

**Rule of thumb:**
- Treat as number (ordinal) if the gaps are roughly equal and you have limited data
- Treat as one-hot if the gaps are clearly unequal and you have enough data per level

For the gradient:
- Discrete as number: contribution and vote scale with the count
- Discrete as one-hot: only the active level column contributes; all others are silent

In [ ]:
# ── Observation 42: num_children and education — two encoding strategies ──────

obs_children  = int(df['num_children'].iloc[OBS])
obs_education = int(df['education'].iloc[OBS])

# ── Numeric (ordinal) encoding — weight × count/level ─────────────────────────
w_children_num = TRUE_W['num_children']
w_education_num = TRUE_W['education']

contrib_children_num  = w_children_num  * obs_children
contrib_education_num = w_education_num * obs_education

# ── One-hot encoding — one column per level; only the active level speaks ──────
children_levels  = [0, 1, 2, 3, 4]
education_levels = [1, 2, 3, 4]
edu_label        = {1: 'HS', 2: 'Bachelor', 3: 'Master', 4: 'PhD'}

# Fit separate models with one-hot encoded versions to get per-level weights
children_onehot = pd.get_dummies(df['num_children'], prefix='ch').values
edu_onehot      = pd.get_dummies(df['education'],    prefix='edu').values

from sklearn.linear_model import LinearRegression as _LR
X_ch  = children_onehot[:, 1:]     # drop level-0 column (baseline)
X_edu = edu_onehot[:, 1:]          # drop level-1 column (baseline)

w_ch_oh  = _LR().fit(X_ch,  df['credit_limit']).coef_
w_edu_oh = _LR().fit(X_edu, df['credit_limit']).coef_

# ── Print comparison ───────────────────────────────────────────────────────────
print(f"Observation {OBS}:  num_children = {obs_children}   education = {obs_education} ({edu_label[obs_education]})")
print()
print("── num_children ──────────────────────────────────────────────────────────")
print(f"  Numeric (ordinal):  w={w_children_num:.2f}  x  value={obs_children}")
print(f"    contribution = {contrib_children_num:.2f}k   {'(feature is SILENT — count is 0)' if obs_children==0 else ''}")
print()
print("  One-hot (separate level per count):")
for i, lv in enumerate(children_levels[1:]):
    active = (lv == obs_children)
    wt     = w_ch_oh[i]
    val    = 1 if active else 0
    c      = wt * val
    tag    = " <- ACTIVE" if active else "   silent"
    print(f"    ch_{lv}:  w={wt:>6.2f}  x  value={val}  = {c:>6.2f}k{tag}")

print()
print("── education ─────────────────────────────────────────────────────────────")
print(f"  Numeric (ordinal):  w={w_education_num:.2f}  x  level={obs_education} ({edu_label[obs_education]})")
print(f"    contribution = {contrib_education_num:.2f}k")
print()
print("  One-hot (separate level per education tier):")
for i, lv in enumerate(education_levels[1:]):
    active = (lv == obs_education)
    wt     = w_edu_oh[i]
    val    = 1 if active else 0
    c      = wt * val
    tag    = f" <- ACTIVE ({edu_label[lv]})" if active else f"   silent ({edu_label[lv]})"
    print(f"    edu_{lv}:  w={wt:>6.2f}  x  value={val}  = {c:>6.2f}k{tag}")

# ── Visualisation ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

# Top row: num_children
ch_num_vals = [w_children_num * v for v in children_levels]
ch_num_cols = ['#1565C0' if v == obs_children else ('#e57373' if w_children_num * v < 0 else '#4CAF50')
               for v in children_levels]
axes[0,0].bar([str(v) for v in children_levels], ch_num_vals, color=ch_num_cols, edgecolor='white')
axes[0,0].axhline(0, color='black', lw=1)
axes[0,0].set_title(f'num_children — numeric encoding\ncontribution = weight x count   (blue = obs {OBS})', fontsize=10)
axes[0,0].set_xlabel('num_children value')
axes[0,0].set_ylabel('Contribution (k$)')

ch_oh_vals = [0] + list(w_ch_oh)
ch_oh_cols = ['#1565C0' if v == obs_children else ('#e57373' if w < 0 else '#4CAF50')
              for v, w in zip(children_levels, ch_oh_vals)]
axes[0,1].bar([str(v) for v in children_levels], ch_oh_vals, color=ch_oh_cols, edgecolor='white')
axes[0,1].axhline(0, color='black', lw=1)
axes[0,1].set_title(f'num_children — one-hot encoding\neach level has its own weight   (blue = obs {OBS})', fontsize=10)
axes[0,1].set_xlabel('num_children level')
axes[0,1].set_ylabel('Contribution (k$)')

# Bottom row: education
edu_num_vals = [w_education_num * v for v in education_levels]
edu_num_cols = ['#1565C0' if v == obs_education else '#4CAF50' for v in education_levels]
axes[1,0].bar([edu_label[v] for v in education_levels], edu_num_vals, color=edu_num_cols, edgecolor='white')
axes[1,0].axhline(0, color='black', lw=1)
axes[1,0].set_title(f'education — numeric encoding\nequal gap assumed between levels   (blue = obs {OBS})', fontsize=10)
axes[1,0].set_xlabel('education level')
axes[1,0].set_ylabel('Contribution (k$)')

edu_oh_vals = [0] + list(w_edu_oh)
edu_oh_cols = ['#1565C0' if v == obs_education else '#4CAF50' for v in education_levels]
axes[1,1].bar([edu_label[v] for v in education_levels], edu_oh_vals, color=edu_oh_cols, edgecolor='white')
axes[1,1].axhline(0, color='black', lw=1)
axes[1,1].set_title(f'education — one-hot encoding\neach level has its own weight   (blue = obs {OBS})', fontsize=10)
axes[1,1].set_xlabel('education level')
axes[1,1].set_ylabel('Contribution (k$)')

plt.suptitle('Numeric vs one-hot encoding: same feature, different assumptions about the gaps',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print()
print("Numeric:  assumes equal spacing — 0->1 child is same credit impact as 3->4 children")
print("One-hot:  no such assumption — each level gets its own weight learned from data")
print()
print("In this synthetic dataset the true weights ARE linear, so numeric ≈ one-hot.")
print("On real data (unequal gaps), one-hot is more flexible but needs enough data per level.")

## Key Takeaways

### Correlation measure by feature type

| Feature type | Measure to use | Why |
|---|---|---|
| Continuous | Pearson r | Measures linear proportional relationship |
| Discrete / Ordinal | Spearman ρ | Uses ranks — works when gaps are unequal |
| Binary | Point-biserial | Compares target mean across the two groups |
| Categorical | Pearson per dummy OR mutual information | Each encoded column is treated as binary |

### For one observation — contribution = weight × value

| Feature type | When silent (contribution = 0) | When active |
|---|---|---|
| Continuous | Only if value = 0 (rare) | Always — large values are loud |
| Discrete | When count = 0 | Proportional to the count |
| Ordinal | When level = 0 (unusual) | Higher level → larger contribution |
| Binary | **When value = 0** (feature absent) | When value = 1 — full weight passes through |
| Categorical dummy | **Inactive categories (value=0)** | Only the one active category speaks |

### For the gradient — error × feature value

```
gradient vote for weight j  =  error  ×  feature_value_j
```

- Feature value = 0 → **zero gradient** — this row cannot change that weight
- Raw features: scale drives the vote size, not importance
- Scaled features: vote is proportional to how unusual this person's value is

### The fundamental insight

Every feature — regardless of type — contributes through the same arithmetic: **weight × value**.
The type of the feature determines what values it can take, and therefore when it speaks and when it stays silent.
Binary and categorical features are designed to go silent for observations where they do not apply.